# Track B - full run: flood & damage mapping from space
Run the cells **top to bottom**. Only the CONFIG cell changes between events.

| Step | What it does | Time |
|---|---|---|
| 1 CONFIG | area, flood date, OSM snapshot, ohsome key | - |
| 2 Login | Copernicus Data Space (openEO) | 1 min |
| 3 Model | upload `flood_unet.onnx` + `norm.json` | 1 min |
| 4 Mapping | Sentinel-1 asc+desc, DEM masks, U-Net + debris change, fused impact | 15-20 min |
| 5 OSM | pre-event buildings/roads/bridges/settlements/health (ohsome v2) | 3-5 min |
| 6 Damage | what was hit | 1-3 min |
| 7 Access | who is cut off + repair priority | 1-5 min |
| 8 Save | zip + download all outputs | 1 min |

Inputs: Sentinel-1/2 + Copernicus DEM (CDSE), pre-event OSM (ohsome), model trained on Kuro Siwo. **No EMS/UNOSAT/post-event OSM.**

In [ ]:
# 1) CONFIG - change only these for a new event
import os, getpass, datetime as dt
BBOX = [85.10, 27.90, 85.45, 28.32]          # W, S, E, N
EVENT = "2026-08-26"                         # flood date
OSM_SNAPSHOT = "2026-07-27"                  # pre-event OSM date (Trishuli: as given in the rules). Default: event - 1 day
# OSM_SNAPSHOT = str(dt.date.fromisoformat(EVENT) - dt.timedelta(days=1))
assert dt.date.fromisoformat(OSM_SNAPSHOT) < dt.date.fromisoformat(EVENT)
if not os.environ.get("OHSOME_API_KEY"):
    os.environ["OHSOME_API_KEY"] = getpass.getpass("ohsome API key (from account.heigit.org): ").strip()
print("Config OK:", BBOX, EVENT, "OSM snapshot", OSM_SNAPSHOT)

In [ ]:
# 2) Copernicus Data Space login (click the link, press Allow)
!pip install -q openeo rasterio
import openeo
con = openeo.connect("openeo.dataspace.copernicus.eu").authenticate_oidc(max_poll_time=900)
print("Logged in:", con.describe_account().get("name", "ok"))

In [ ]:
# 3) Model files - upload flood_unet.onnx + norm.json (skipped if already here)
import os
if not (os.path.exists("flood_unet.onnx") and os.path.exists("norm.json")):
    from google.colab import files
    files.upload()
    for f in os.listdir():
        if f.startswith("flood_unet") and f.endswith(".onnx") and f != "flood_unet.onnx": os.rename(f, "flood_unet.onnx")
        if f.startswith("norm") and f != "norm.json" and (f.endswith(".json") or f.endswith(".txt")): os.rename(f, "norm.json")
print("onnx:", os.path.exists("flood_unet.onnx"), "| norm:", os.path.exists("norm.json"))

In [ ]:
# 4) MAPPING PIPELINE
# Track B - mapping pipeline v1: area + flood date -> fused impact map (ascending + descending Sentinel-1)
# Needs in the Colab session: `con` (openEO login) and flood_unet.onnx + norm.json uploaded.
# Inputs used: Sentinel-1 GRD + Copernicus DEM GLO-30 only (allowed sources). No EMS / post-event OSM.
!pip install -q openeo rasterio pysheds onnxruntime scipy

import os, json, time, datetime as dt
from collections import defaultdict
import numpy as np, requests, rasterio, onnxruntime as ort, matplotlib.pyplot as plt
from rasterio.warp import reproject, Resampling
from scipy.ndimage import uniform_filter, label
from matplotlib.colors import ListedColormap
from pysheds.grid import Grid

# BBOX and EVENT come from the CONFIG cell
# ===============================================================
SEARCH_DAYS = 36                          # look this far before/after the event for S1 scenes
MAX_PAIR_GAP = 48                         # days between pre and post (12 or 24 typical)
MIN_COVERAGE = 0.6                        # a pair must cover >= 60% of the area to be preferred
HAND_MAX_M, SLOPE_MAX_DEG = 50, 30        # valley-floor mask (tuned on Trishuli)
K_SIGMA, MIN_DB, MIN_BLOB_PX = 2.5, 2.0, 50
TILE, STRIDE = 224, 160
missing = [f for f in ("flood_unet.onnx", "norm.json") if not os.path.exists(f)]
assert not missing, f"Upload {missing} first (run the upload cell), then re-run this cell"
assert "con" in globals(), "Run the openEO login cell first (con = openeo.connect(...).authenticate_oidc())"
SAVE_TO_DRIVE = False                    # keep results in Google Drive so a Colab disconnect loses nothing
RUN_NAME = "run_" + EVENT.replace("-", "") + "_" + "_".join(f"{v:.2f}" for v in BBOX)
OUT = RUN_NAME
if SAVE_TO_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        OUT = f"/content/drive/MyDrive/trackb/{RUN_NAME}"
    except Exception as e:
        print("Drive not mounted, saving locally:", e)
os.makedirs(OUT, exist_ok=True)
for name in ("s1_orbit85_ascending.tif", "s1_orbit121_descending.tif", "dem30.tif"):   # reuse local copies from an earlier run
    if os.path.exists(f"{RUN_NAME}/{name}") and not os.path.exists(f"{OUT}/{name}"):
        import shutil; shutil.copy(f"{RUN_NAME}/{name}", f"{OUT}/{name}"); print("copied", name, "to", OUT)
ODATA = "https://catalogue.dataspace.copernicus.eu/odata/v1/Products"
log = {"bbox": BBOX, "event": EVENT, "sources": ["Sentinel-1 GRD (CDSE)", "Copernicus DEM GLO-30"], "orbits": []}


def say(msg):
    print(f"[{dt.datetime.now():%H:%M:%S}] {msg}", flush=True)


def utm_epsg(bbox):
    lon, lat = (bbox[0] + bbox[2]) / 2, (bbox[1] + bbox[3]) / 2
    return (32600 if lat >= 0 else 32700) + int((lon + 180) // 6) + 1


# ---------- 1. find same-orbit pre/post pairs ----------
def find_pairs(bbox, event):
    ev = dt.date.fromisoformat(event)
    w, s, e, n = bbox
    wkt = f"POLYGON(({w} {s},{e} {s},{e} {n},{w} {n},{w} {s}))"
    flt = ("Collection/Name eq 'SENTINEL-1' and contains(Name,'_IW_GRDH_1SDV_') "
           f"and OData.CSC.Intersects(area=geography'SRID=4326;{wkt}') "
           f"and ContentDate/Start gt {(ev - dt.timedelta(SEARCH_DAYS)).isoformat()}T00:00:00.000Z "
           f"and ContentDate/Start lt {(ev + dt.timedelta(SEARCH_DAYS)).isoformat()}T23:59:59.000Z")
    params = {"$filter": flt, "$expand": "Attributes", "$top": 200}
    items, url = [], ODATA
    while url:
        r = requests.get(url, params=params, timeout=120); r.raise_for_status(); js = r.json()
        items += js.get("value", []); url, params = js.get("@odata.nextLink"), None
    from shapely.geometry import shape, box
    from shapely.ops import unary_union
    aoi = box(w, s, e, n)
    foot = defaultdict(list)                          # (orbit, direction, date) -> footprints of all slices
    for p in items:
        if p["Name"].endswith("_COG.SAFE"):
            continue
        a = {x["Name"]: x.get("Value") for x in p.get("Attributes", [])}
        d = dt.date.fromisoformat(p["ContentDate"]["Start"][:10])
        try:
            g = shape(p["GeoFootprint"])
        except Exception:
            g = aoi                                    # footprint missing -> assume full cover
        foot[(a.get("relativeOrbitNumber"), str(a.get("orbitDirection")).upper(), d)].append(g)
    cover = {k: unary_union(v).intersection(aoi).area / aoi.area for k, v in foot.items()}
    groups = defaultdict(dict)                         # (orbit, direction) -> {date: coverage}
    for (orbit, direction, d), c in cover.items():
        groups[(orbit, direction)][d] = c
    pairs = []
    for (orbit, direction), dates in groups.items():
        pre = [d for d in dates if d < ev]; post = [d for d in dates if d >= ev]
        if pre and post and (min(post) - max(pre)).days <= MAX_PAIR_GAP:
            cov = min(dates[max(pre)], dates[min(post)])
            pairs.append({"orbit": orbit, "direction": direction, "pre": max(pre), "post": min(post),
                          "gap_days": (min(post) - max(pre)).days, "days_after_event": (min(post) - ev).days,
                          "aoi_coverage": round(cov, 2)})
    # prefer pairs that cover most of the area, then the earliest post-event image
    pairs.sort(key=lambda p: (p["aoi_coverage"] < MIN_COVERAGE, p["days_after_event"], -p["aoi_coverage"]))
    chosen = {}
    for p in pairs:                                   # best ascending + best descending
        chosen.setdefault(p["direction"], p)
    return list(chosen.values()), pairs


# ---------- 2. openEO: calibrated dB stack per pair (jobs run in parallel) ----------
def s1_cube(bbox, pair, epsg):
    ext = dict(zip(["west", "south", "east", "north"], bbox))
    def one(day, prefix):
        c = con.load_collection("SENTINEL1_GRD", spatial_extent=ext, bands=["VV", "VH"],
                                temporal_extent=[day.isoformat(), (day + dt.timedelta(1)).isoformat()])
        c = c.sar_backscatter(coefficient="sigma0-ellipsoid").resample_spatial(resolution=10, projection=epsg)
        c = c.reduce_dimension(dimension="t", reducer="mean").apply(lambda x: x.log(base=10) * 10)
        return c.rename_labels("bands", [f"{prefix}_VV", f"{prefix}_VH"])
    return one(pair["pre"], "pre").merge_cubes(one(pair["post"], "post"))


def with_retry(fn, what, tries=6):
    """Retry openEO calls on 'Too Many Requests' (429) and short network hiccups, waiting longer each time."""
    for k in range(1, tries + 1):
        try:
            return fn()
        except Exception as e:
            busy = "429" in str(e) or "Too Many Requests" in str(e)
            net = isinstance(e, (requests.exceptions.ConnectionError, requests.exceptions.Timeout))
            if (busy or net) and k < tries:
                wait = 30 * k
                say(f"   {what}: server busy ({type(e).__name__}) - retry {k}/{tries - 1} in {wait}s")
                time.sleep(wait)
            else:
                raise


def run_jobs(bbox, pairs, epsg):
    """One openEO batch job at a time (free tier limits parallel jobs), with retries and caching."""
    for p in pairs:
        path = f"{OUT}/s1_orbit{p['orbit']}_{p['direction'].lower()}.tif"
        p["file"] = path
        if os.path.exists(path):
            say(f"cached {path}"); continue
        tag = f"orbit {p['orbit']} {p['direction']}"
        try:
            job = with_retry(lambda: s1_cube(bbox, p, epsg).create_job(
                title=f"TrackB S1 orbit {p['orbit']}", out_format="GTiff"), f"create job {tag}")
            with_retry(job.start, f"start job {tag}")
            say(f"started openEO job for {tag} ({p['pre']} -> {p['post']})")
            while True:
                time.sleep(20)
                st = with_retry(job.status, f"status {tag}")
                if st == "finished":
                    with_retry(lambda: job.get_results().download_file(path), f"download {tag}")
                    say(f"downloaded {path}"); break
                if st in ("error", "canceled"):
                    say(f"!! job for {tag} ended with '{st}' - continuing without this orbit"); break
        except Exception as e:
            say(f"!! {tag} failed: {type(e).__name__}: {str(e)[:200]} - continuing without this orbit")
    return [p for p in pairs if os.path.exists(p["file"])]


# ---------- grid helpers ----------
def read(path):
    with rasterio.open(path) as s:
        return s.read().astype("float32"), s.transform, s.crs, s.profile


def to_grid(arr, src_tr, src_crs, ref, resampling=Resampling.bilinear):
    ref_tr, ref_crs, shape = ref
    out = np.full((arr.shape[0],) + shape, np.nan, dtype="float32")
    for i in range(arr.shape[0]):
        reproject(arr[i], out[i], src_transform=src_tr, src_crs=src_crs, dst_transform=ref_tr, dst_crs=ref_crs,
                  resampling=resampling, src_nodata=np.nan, dst_nodata=np.nan)
    return out


# ---------- 3. terrain: DEM -> slope + HAND ----------
def terrain(bbox, epsg, ref):
    dem_path = f"{OUT}/dem30.tif"
    if not os.path.exists(dem_path):
        ext = dict(zip(["west", "south", "east", "north"], bbox))
        (con.load_collection("COPERNICUS_30", spatial_extent=ext).reduce_dimension(dimension="t", reducer="max")
            .resample_spatial(resolution=30, projection=epsg)).download(dem_path, format="GTiff")
    grid = Grid.from_raster(dem_path); dem = grid.read_raster(dem_path)
    cond = grid.resolve_flats(grid.fill_depressions(grid.fill_pits(dem)))
    fdir = grid.flowdir(cond); acc = grid.accumulation(fdir)
    hand30 = np.asarray(grid.compute_hand(fdir, cond, acc > 2000), dtype="float32")
    d30, tr, crs, _ = read(dem_path)
    gy, gx = np.gradient(d30[0], abs(tr.e), tr.a)
    slope30 = np.degrees(np.arctan(np.hypot(gx, gy))).astype("float32")
    stack = to_grid(np.stack([d30[0], slope30, hand30]), tr, crs, ref)
    return stack[0], stack[1], stack[2]


# ---------- 4. per-orbit detection ----------
NORM = json.load(open("norm.json"))
SESS = ort.InferenceSession("flood_unet.onnx", providers=["CPUExecutionProvider"])


def make_input(sar_db, dem, norm=NORM):
    dem = np.nan_to_num(dem, nan=float(np.nanmean(dem)) if np.isfinite(dem).any() else 0.0)
    lo, hi = norm["sar_clip_db"]
    sar = (np.clip(sar_db, lo, hi) + norm["sar_shift"]) / norm["sar_scale"]
    gy, gx = np.gradient(dem, norm["pixel_size_m"])
    slp = np.degrees(np.arctan(np.hypot(gx, gy))) / norm["slope_scale_deg"]
    dem_rel = np.clip((dem - dem.mean()) / norm["dem_rel_scale_m"], -5, 5)
    return np.concatenate([sar, dem_rel[None], slp[None]]).astype(np.float32)


def unet_classes(sar_db, dem10):
    H, W = dem10.shape
    probs = np.zeros((3, H, W), np.float32); cnt = np.zeros((H, W), np.float32)
    rows = list(range(0, max(H - TILE, 0) + 1, STRIDE)); cols = list(range(0, max(W - TILE, 0) + 1, STRIDE))
    if rows[-1] != H - TILE: rows.append(H - TILE)
    if cols[-1] != W - TILE: cols.append(W - TILE)
    t0 = time.time()
    for i, r in enumerate(rows):
        if i % 5 == 0:
            done = i / len(rows)
            eta = (time.time() - t0) / done * (1 - done) / 60 if done else float("nan")
            say(f"   U-Net row {i}/{len(rows)} ({done:.0%}) - about {eta:.0f} min left")
        batch = np.stack([make_input(sar_db[:, r:r + TILE, c:c + TILE], dem10[r:r + TILE, c:c + TILE]) for c in cols])
        lg = SESS.run(None, {"x": batch})[0]
        e = np.exp(lg - lg.max(1, keepdims=True)); p = e / e.sum(1, keepdims=True)
        for k, c in enumerate(cols):
            probs[:, r:r + TILE, c:c + TILE] += p[k]; cnt[r:r + TILE, c:c + TILE] += 1
    return (probs / np.maximum(cnt, 1)).argmax(0)


def despeckle_db(db, size=7):
    return 10 * np.log10(np.clip(uniform_filter(10 ** (db / 10), size), 1e-6, None))


def drop_small(mask):
    lab, n = label(mask)
    if n == 0: return mask
    sz = np.bincount(lab.ravel()); keep = sz >= MIN_BLOB_PX; keep[0] = False
    return keep[lab]


def detect(stack, valley, dem10, pair):
    valid = np.all(np.isfinite(stack) & (stack > -45), axis=0)
    sar = np.where(np.isfinite(stack), stack, -30.0)
    region = valley & valid
    res = {"valid": valid}
    thr = {}
    for pol, (a, b) in {"VV": (0, 2), "VH": (1, 3)}.items():
        diff = despeckle_db(sar[b]) - despeckle_db(sar[a])
        diff -= np.median(diff[region])
        mad = np.median(np.abs(diff[region])) * 1.4826
        thr[pol] = max(MIN_DB, K_SIGMA * mad); res[pol] = diff
    dark = region & ((res["VV"] < -thr["VV"]) | (res["VH"] < -thr["VH"]))
    bright = region & ((res["VV"] > thr["VV"]) | (res["VH"] > thr["VH"])) & ~dark
    say(f"orbit {pair['orbit']}: running U-Net on {sar.shape[1]}x{sar.shape[2]} px")
    water = (unet_classes(sar, dem10) == 2) & region
    pair["thresholds_db"] = {k: round(float(v), 2) for k, v in thr.items()}
    return {"valid": valid, "water": water, "dark": drop_small(dark), "bright": drop_small(bright)}


# ======================= RUN =======================
say(f"Searching Sentinel-1 pairs for {BBOX}, event {EVENT}")
chosen, all_pairs = find_pairs(BBOX, EVENT)
for p in all_pairs:
    print(f"   orbit {p['orbit']:>4} {p['direction']:<10} pre {p['pre']} post {p['post']} "
          f"gap {p['gap_days']}d  (+{p['days_after_event']}d after event)  covers {p['aoi_coverage']*100:.0f}% of area")
if not chosen:
    raise SystemExit("No same-orbit Sentinel-1 pair found around this date - cannot map this event with radar.")
say("Using: " + ", ".join(f"orbit {p['orbit']} {p['direction']}" for p in chosen))

epsg = utm_epsg(BBOX)
chosen = run_jobs(BBOX, chosen, epsg)
if not chosen:
    raise SystemExit("All openEO jobs failed - see messages above.")

ref_arr, ref_tr, ref_crs, ref_prof = read(chosen[0]["file"])
ref = (ref_tr, ref_crs, ref_arr.shape[1:])
say("Building terrain masks (Copernicus DEM)")
dem10, slope, hand = terrain(BBOX, epsg, ref)
valley = (hand < HAND_MAX_M) & (slope < SLOPE_MAX_DEG)

per_orbit = []
for p in chosen:
    arr, tr, crs, _ = read(p["file"])
    stack = arr if p is chosen[0] else to_grid(arr, tr, crs, ref)
    per_orbit.append(detect(stack, valley, dem10, p))

# ---------- 5. fuse orbits (observability-aware) ----------
# An orbit only counts where it could actually see the ground (radar shadow / layover / swath edge = no data).
valids = np.array([o["valid"] for o in per_orbit])
dets = np.array([(o["water"] | o["dark"] | o["bright"]) for o in per_orbit]) & valids
n_valid = valids.sum(0)
n_det = dets.sum(0)
any_valid = n_valid > 0
water = np.any([o["water"] & o["valid"] for o in per_orbit], axis=0)
dark = np.any([o["dark"] & o["valid"] for o in per_orbit], axis=0) & ~water
bright = np.any([o["bright"] & o["valid"] for o in per_orbit], axis=0) & ~water & ~dark
hit = water | dark | bright

# confidence: 3 = seen by >=2 orbits, all agree | 2 = only one orbit could see it, and it saw damage
#             1 = >=2 orbits could see it, they disagree | 0 = no damage | 255 = outside valley / unobserved
conf = np.zeros(valley.shape, np.uint8)
conf[hit & (n_valid >= 2) & (n_det == n_valid)] = 3
conf[hit & (n_valid == 1)] = 2
conf[hit & (n_valid >= 2) & (n_det < n_valid)] = 1

impact = np.zeros(valley.shape, np.uint8)
impact[dark] = 2; impact[bright] = 3; impact[water] = 1
impact[~valley | ~any_valid] = 255
conf[impact == 255] = 255
prof = ref_prof.copy(); prof.update(count=2, dtype="uint8", nodata=255, compress="deflate")
with rasterio.open(f"{OUT}/impact_fused.tif", "w", **prof) as dst:
    dst.write(impact, 1); dst.write(conf, 2)
    dst.update_tags(1, classes="0=unchanged,1=new_water_unet,2=debris_darkened,3=debris_brightened,255=outside_valley_or_unobserved")
    dst.update_tags(2, confidence="3=high(all observing orbits agree),2=medium(single view),1=low(orbits disagree),0=no_damage,255=n/a")

px = abs(ref_tr.a * ref_tr.e) / 1e6
log["orbits"] = []
stats = {"new_water_km2": water.sum() * px, "debris_darkened_km2": dark.sum() * px,
         "debris_brightened_km2": bright.sum() * px, "total_impacted_km2": hit.sum() * px,
         "conf_high_km2": (conf == 3).sum() * px, "conf_medium_single_view_km2": (conf == 2).sum() * px,
         "conf_low_disagree_km2": (conf == 1).sum() * px,
         "valley_floor_km2": valley.sum() * px,
         "valley_seen_by_2plus_orbits_km2": (valley & (n_valid >= 2)).sum() * px,
         "valley_seen_by_1_orbit_km2": (valley & (n_valid == 1)).sum() * px,
         "valley_unobserved_km2": (valley & (n_valid == 0)).sum() * px}
for k, o in enumerate(per_orbit):
    stats[f"valley_visible_orbit{chosen[k]['orbit']}_pct"] = o["valid"][valley].mean() * 100
stats = {k: round(float(v), 3) for k, v in stats.items()}
for p in chosen:
    log["orbits"].append({k: (str(v) if isinstance(v, dt.date) else v) for k, v in p.items()})
log["stats"] = stats
log["params"] = {"hand_max_m": HAND_MAX_M, "slope_max_deg": SLOPE_MAX_DEG, "k_sigma": K_SIGMA,
                 "min_db": MIN_DB, "min_blob_ha": MIN_BLOB_PX * px * 100}
json.dump(log, open(f"{OUT}/run_log.json", "w"), indent=2)

print("\n================ SUMMARY ================")
for p in chosen:
    print(f"orbit {p['orbit']} {p['direction']}: {p['pre']} -> {p['post']} | thresholds {p['thresholds_db']}")
for k, v in stats.items():
    print(f"{k:<24} {v}")
print(f"Saved {OUT}/impact_fused.tif and {OUT}/run_log.json")

# ---------- 6. figure ----------
b = rasterio.transform.array_bounds(*ref[2], ref_tr)
ext = [b[0] / 1000, b[2] / 1000, b[1] / 1000, b[3] / 1000]
bg = despeckle_db(np.where(np.isfinite(ref_arr[2]), ref_arr[2], -30))
cmap = ListedColormap(["#1f6feb", "#8b4513", "#f0a020"])
fig, ax = plt.subplots(2, 2, figsize=(18, 20))

def draw(axis, cls, title):
    axis.imshow(bg, extent=ext, cmap="gray", vmin=-25, vmax=0)
    axis.imshow(np.where(np.isin(cls, [1, 2, 3]), cls, np.nan), extent=ext, cmap=cmap, vmin=0.5, vmax=3.5,
                interpolation="nearest")
    axis.set_title(title)

for k, (o, p) in enumerate(zip(per_orbit[:2], chosen[:2])):
    c = np.zeros_like(impact); c[o["dark"]] = 2; c[o["bright"]] = 3; c[o["water"]] = 1
    draw(ax[0, k], c, f"orbit {p['orbit']} {p['direction']} only ({p['pre']} -> {p['post']})")
if len(per_orbit) == 1:
    ax[0, 1].axis("off")
draw(ax[1, 0], impact, "FUSED impact (any orbit)")
ax[1, 1].imshow(bg, extent=ext, cmap="gray", vmin=-25, vmax=0)
ax[1, 1].imshow(np.where(valley & (n_valid == 0), 1.0, np.nan), extent=ext, cmap=ListedColormap(["#7a00a0"]),
                vmin=0, vmax=2, alpha=0.5, interpolation="nearest")
ax[1, 1].imshow(np.where(np.isin(conf, [1, 2, 3]), conf, np.nan), extent=ext,
                cmap=ListedColormap(["#999999", "#f0c000", "#d00000"]), vmin=0.5, vmax=3.5, interpolation="nearest")
ax[1, 1].set_title("Confidence: red = orbits agree, yellow = only 1 orbit could see,\n"
                   "grey = orbits disagree, purple = valley no orbit could see")
handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in ["#1f6feb", "#8b4513", "#f0a020"]]
ax[1, 0].legend(handles, ["new water (U-Net)", "debris - darkened", "debris - brightened"], loc="upper left")
for axis in ax.ravel():
    axis.set_xlabel("UTM easting (km)"); axis.set_ylabel("UTM northing (km)")
plt.tight_layout(); plt.savefig(f"{OUT}/impact_fused.png", dpi=110); plt.show()
print("Contains modified Copernicus Sentinel data 2026.")
print("Produced using Copernicus WorldDEM-30 (c) DLR e.V. 2010-2014 and (c) Airbus Defence and Space GmbH 2014-2018 "
      "provided under COPERNICUS by the European Union and ESA; all rights reserved.")



In [ ]:
# 5) PRE-EVENT OSM (ohsome API v2)
!pip install -q geopandas pyarrow pyproj pyogrio
"""
Track B - download PRE-EVENT OpenStreetMap layers with the ohsome API v2 (v1 extraction is now refused: HTTP 403).
Runs on a laptop OR Colab. Needs a free ohsome API key: https://account.heigit.org/signup

Setup once:   pip install requests geopandas pyarrow
Run:          python osm_fetch_v2.py          (it asks for the API key; or set env var OHSOME_API_KEY)
Output:       folder osm_pre_event/ with 6 GeoJSON files (same names/format osm_damage.py expects)

Data rule: snapshot timestamp 2026-07-27 (latest pre-event date named in the rules) - never "latest".
"""
import datetime as dt
import getpass
import io
import os
import time

import geopandas as gpd
import pandas as pd
import requests

SNAPSHOT = OSM_SNAPSHOT                        # from the CONFIG cell
URL = "https://api.heigit.org/ohsome-api/v2-rc/extraction/features.parquet"
assert dt.date.fromisoformat(SNAPSHOT) < dt.date.fromisoformat(EVENT), "snapshot must be BEFORE the event"

LAYERS = {   # name: (ohsome filter, tiles per side)  - server timeout is 4 min, so dense layers use more tiles
    "buildings":   ("building=* and building!=no and geometry:polygon", 2),
    "roads":       ("highway in (motorway,trunk,primary,secondary,tertiary,unclassified,residential,service,track,"
                    "motorway_link,trunk_link,primary_link,secondary_link,tertiary_link) and geometry:line", 2),
    "footpaths":   ("highway in (path,footway,steps,bridleway) and geometry:line", 2),
    "bridges":     ("bridge=* and bridge!=no and geometry:line", 1),
    "settlements": ("place in (city,town,village,hamlet,isolated_dwelling)", 1),
    "health":      ("amenity in (hospital,clinic,doctors) or healthcare=*", 1),
}


def say(msg):
    print(f"[{dt.datetime.now():%H:%M:%S}] {msg}", flush=True)


def bbox_polygon(bb):
    w, s, e, n = bb
    return {"type": "Polygon", "coordinates": [[[w, s], [e, s], [e, n], [w, n], [w, s]]]}


def to_gdf(content):
    try:
        g = gpd.read_parquet(io.BytesIO(content), to_pandas_kwargs={"maps_as_pydicts": "strict"})
    except TypeError:                                   # older geopandas without to_pandas_kwargs
        g = gpd.read_parquet(io.BytesIO(content))
    if "tags" in g.columns:
        tags = g["tags"].apply(lambda t: dict(t) if t is not None else {})
        g = g.drop(columns="tags").join(pd.json_normalize(tags.tolist()).set_index(g.index))
    return g


PAUSE_S = 12            # polite gap between requests (free key has a rate limit)


def get_tile(name, flt, bb, key, depth=0):
    """One tile: cached on disk, polite pacing, waits on 429 (rate limit), splits only on timeouts/5xx."""
    label = f"{name} [{bb[0]:.3f},{bb[1]:.3f},{bb[2]:.3f},{bb[3]:.3f}]"
    cache = f"{OUT}/_tiles/{name}_{bb[0]:.4f}_{bb[1]:.4f}_{bb[2]:.4f}_{bb[3]:.4f}.parquet"
    if os.path.exists(cache):
        return [to_gdf(open(cache, "rb").read())]
    payload = {"aoi": bbox_polygon(bb), "filter": flt, "time": f"{SNAPSHOT}T00:00:00Z", "clip": False}
    fails = 0
    while True:
        time.sleep(PAUSE_S)
        try:
            r = requests.post(URL, json=payload, headers={"authorization": key}, timeout=300)
        except requests.exceptions.RequestException as ex:
            say(f"   {label}: {type(ex).__name__}"); fails += 1
            if fails >= 2: break
            continue
        if r.status_code == 200:
            os.makedirs(f"{OUT}/_tiles", exist_ok=True)
            with open(cache, "wb") as fh:
                fh.write(r.content)
            say(f"   {label}: ok")
            return [to_gdf(r.content)] if r.content else []
        body = r.text[:200].replace("\n", " ")
        if r.status_code == 429:                                  # rate limit: wait, do NOT split (would make it worse)
            wait = int(r.headers.get("Retry-After", "0") or 0) or 90
            say(f"   {label}: rate limit - waiting {wait}s"); time.sleep(wait); continue
        say(f"   {label}: HTTP {r.status_code} - {body}")
        if r.status_code in (400, 401, 403, 404, 422):
            raise SystemExit(f"Request refused (HTTP {r.status_code}). Check the API key / send this message to Claude.")
        fails += 1
        if fails >= 2: break
        time.sleep(30)
    if depth >= 3:
        raise SystemExit(f"Kept failing for {label} - try again later")
    say(f"   splitting {label} into 4 (server timeout)")
    mx, my = (bb[0] + bb[2]) / 2, (bb[1] + bb[3]) / 2
    out = []
    for sub in [(bb[0], bb[1], mx, my), (mx, bb[1], bb[2], my), (bb[0], my, mx, bb[3]), (mx, my, bb[2], bb[3])]:
        out += get_tile(name, flt, sub, key, depth + 1)
    return out


def main():
    key = os.environ.get("OHSOME_API_KEY") or getpass.getpass("Paste your ohsome API key (hidden) and press Enter: ").strip()
    os.makedirs(OUT, exist_ok=True)
    say(f"ohsome API v2 - snapshot {SNAPSHOT} (event {EVENT}), bbox {BBOX}")
    for name, (flt, tiles) in LAYERS.items():
        path = f"{OUT}/osm_{name}.geojson"
        if os.path.exists(path):
            say(f"cached {name}"); continue
        w, s, e, n = BBOX
        xs = [w + (e - w) * k / tiles for k in range(tiles + 1)]
        ys = [s + (n - s) * k / tiles for k in range(tiles + 1)]
        parts = []
        for i in range(tiles):
            for j in range(tiles):
                parts += get_tile(name, flt, (xs[i], ys[j], xs[i + 1], ys[j + 1]), key)
        parts = [p for p in parts if len(p)]
        if parts:
            geom_col = parts[0].geometry.name                 # v2 parquet may call it "geometry" or "geom"
            g = gpd.GeoDataFrame(pd.concat(parts, ignore_index=True), geometry=geom_col,
                                 crs=parts[0].crs or 4326)
            if geom_col != "geometry":
                g = g.drop(columns=["geometry"], errors="ignore").rename_geometry("geometry")
            id_col = next((c for c in ("@osmId", "osm_id", "id") if c in g.columns), None)
            if id_col and id_col != "@osmId":
                g["@osmId"] = g[id_col].astype(str)
            if "@osmId" in g.columns:
                g = g.drop_duplicates("@osmId")
            # keep geometry + simple columns only (GeoJSON can't hold lists/dicts)
            keep = [c for c in g.columns if c == "geometry" or g[c].map(lambda v: isinstance(v, (list, dict))).sum() == 0]
            g = g[keep].to_crs(4326)
        else:
            g = gpd.GeoDataFrame({"@osmId": []}, geometry=[], crs=4326)
        g.to_file(path, driver="GeoJSON")
        say(f"{name}: {len(g)} features -> {path}")
    with open(f"{OUT}/snapshot.txt", "w") as fh:
        fh.write(f"ohsome API v2 snapshot {SNAPSHOT}T00:00:00Z (event {EVENT}), bbox {BBOX}\n(c) OpenStreetMap contributors\n")
    say(f"Done. Upload every osm_*.geojson in '{OUT}' to Colab.")


main()


In [ ]:
# 6) DAMAGE
# Track B - what was damaged: pre-event OpenStreetMap (ohsome) x impact map
# Run in the same Colab session after trishuli_map_v1.py (uses OUT / run folder with impact_fused.tif + run_log.json)
# Data rule: OSM snapshot is ALWAYS before the event date (checked below). No post-event OSM, no EMS/UNOSAT.
!pip install -q geopandas shapely rasterio requests

import os, glob, json, time, datetime as dt
import numpy as np, pandas as pd, requests, rasterio, geopandas as gpd, matplotlib.pyplot as plt
from rasterio.features import shapes, rasterize
from shapely.geometry import shape

RUN = OUT if "OUT" in globals() else sorted(glob.glob("run_*"))[-1]
log = json.load(open(f"{RUN}/run_log.json"))
BBOX, EVENT = log["bbox"], log["event"]
OHSOME = "https://api.ohsome.org/v1"
BUILDING_BUF_M, ROAD_BUF_M, BRIDGE_BUF_M = 10, 10, 30
CONF_NAME = {3: "high", 2: "medium", 1: "low"}
HEADERS = {"User-Agent": "TrackB-flood-mapping/1.0 (student hackathon project; Multimodal AI Hackathon 2026)"}


def say(msg):
    print(f"[{dt.datetime.now():%H:%M:%S}] {msg}", flush=True)


# ---------- 1. pre-event snapshot date (data rule) ----------
if "OSM_SNAPSHOT" in globals():                       # set in the config cell (ohsome v2 snapshot used for download)
    SNAP = OSM_SNAPSHOT
else:
    meta = requests.get(f"{OHSOME}/metadata", headers=HEADERS, timeout=120).json()
    latest = meta["extractRegion"]["temporalExtent"]["toTimestamp"][:10]
    SNAP = str(min(dt.date.fromisoformat(latest), dt.date.fromisoformat(EVENT) - dt.timedelta(days=1)))
assert dt.date.fromisoformat(SNAP) < dt.date.fromisoformat(EVENT), "OSM snapshot must be BEFORE the event"
say(f"pre-event OSM snapshot {SNAP}")

LAYERS = {
    "buildings":   ("building=* and building!=no and geometry:polygon", 3),
    "roads":       ("highway in (motorway,trunk,primary,secondary,tertiary,unclassified,residential,service,track,"
                    "motorway_link,trunk_link,primary_link,secondary_link,tertiary_link) and geometry:line", 2),
    "footpaths":   ("highway in (path,footway,steps,bridleway) and geometry:line", 2),
    "bridges":     ("bridge=* and bridge!=no and geometry:line", 1),
    "settlements": ("place in (city,town,village,hamlet,isolated_dwelling)", 1),
    "health":      ("amenity in (hospital,clinic,doctors) or healthcare=*", 1),
}


def ohsome_post(flt, bb, clip):
    data = {"bboxes": ",".join(f"{v:.5f}" for v in bb), "time": SNAP, "filter": flt, "properties": "tags"}
    if not clip:
        data["clipGeometry"] = "false"
    return requests.post(f"{OHSOME}/elements/geometry", data=data, headers=HEADERS, timeout=600)


def get_tile(name, flt, bb, depth, clip=False):
    """Fetch one tile. Busy/too big -> wait or split into 4 smaller tiles. Bad request -> show why."""
    label = f"{name} [{bb[0]:.3f},{bb[1]:.3f},{bb[2]:.3f},{bb[3]:.3f}]"
    for attempt in range(1, 3):
        try:
            r = ohsome_post(flt, bb, clip)
        except requests.exceptions.RequestException as ex:          # timeout / connection reset
            say(f"   {label}: {type(ex).__name__} (attempt {attempt})"); time.sleep(15 * attempt); continue
        if r.status_code == 200:
            return r.json().get("features", [])
        body = r.text[:300].replace("\n", " ")
        say(f"   {label}: HTTP {r.status_code} - {body}")
        if r.status_code == 400 and not clip:                         # parameter problem -> retry with default clipping
            return get_tile(name, flt, bb, depth, clip=True)
        if r.status_code in (400, 401, 403, 404):
            raise RuntimeError(f"ohsome rejected the request for {label}: HTTP {r.status_code} {body}")
        time.sleep(30 * attempt)                                      # 413 / 429 / 5xx -> wait, then split
    if depth >= 3:
        raise RuntimeError(f"ohsome kept failing for {label} even after splitting - try again later")
    say(f"   splitting {label} into 4 smaller tiles")
    mx, my = (bb[0] + bb[2]) / 2, (bb[1] + bb[3]) / 2
    out = []
    for sub in [(bb[0], bb[1], mx, my), (mx, bb[1], bb[2], my), (bb[0], my, mx, bb[3]), (mx, my, bb[2], bb[3])]:
        out += get_tile(name, flt, sub, depth + 1, clip)
    return out


def fetch(name, flt, tiles):
    """Download one OSM layer from ohsome, tiled + retried + cached. Returns GeoDataFrame (EPSG:4326)."""
    path = f"{RUN}/osm_{name}.geojson"
    if os.path.exists(path):
        say(f"cached {name}"); return gpd.read_file(path)
    w, s, e, n = BBOX
    xs, ys = np.linspace(w, e, tiles + 1), np.linspace(s, n, tiles + 1)
    feats = []
    for i in range(tiles):
        for j in range(tiles):
            feats += get_tile(name, flt, (xs[i], ys[j], xs[i + 1], ys[j + 1]), depth=0)
    if feats:
        gdf = gpd.GeoDataFrame.from_features(feats, crs=4326)
        gdf = gdf.drop_duplicates("@osmId").reset_index(drop=True)
    else:
        gdf = gpd.GeoDataFrame({"@osmId": []}, geometry=[], crs=4326)
    gdf.to_file(path, driver="GeoJSON")
    say(f"{name}: {len(gdf)} features")
    return gdf


osm = {name: fetch(name, flt, tiles) for name, (flt, tiles) in LAYERS.items()}

# ---------- 2. impact map -> polygons with class + confidence ----------
with rasterio.open(f"{RUN}/impact_fused.tif") as src:
    cls, conf = src.read(1), src.read(2)
    tr, crs = src.transform, src.crs
hit = np.isin(cls, [1, 2, 3])
geoms, vals = [], []
for g, v in shapes(cls.astype(np.uint8), mask=hit, transform=tr):
    geoms.append(shape(g)); vals.append(int(v))
imp = gpd.GeoDataFrame({"impact_class": vals}, geometry=geoms, crs=crs)
idx = rasterize(((g, i + 1) for i, g in enumerate(imp.geometry)), out_shape=cls.shape, transform=tr, dtype="int32")
m = idx > 0
conf0 = np.where(conf == 255, 0, conf)
best = pd.DataFrame({"i": idx[m] - 1, "c": conf0[m]}).groupby("i")["c"].max()
imp["conf"] = best.reindex(range(len(imp))).fillna(0).astype(int).values
imp["impact"] = imp["impact_class"].map({1: "new water", 2: "debris (darkened)", 3: "debris (brightened)"})
imp["confidence"] = imp["conf"].map(CONF_NAME).fillna("low")
imp["area_m2"] = imp.area
say(f"impact polygons: {len(imp)}")


def union(gs):
    return gs.union_all() if hasattr(gs, "union_all") else gs.unary_union


def tag_hits(gdf, buf_m):
    """Return copy of gdf (in UTM) with hit flag + best confidence of nearby impact."""
    g = gdf.to_crs(crs).copy()
    if g.empty:
        g["hit"] = False; g["conf"] = 0; return g
    zone = imp[["conf", "geometry"]].copy(); zone["geometry"] = zone.buffer(buf_m)
    j = gpd.sjoin(g[["geometry"]], zone, how="left", predicate="intersects")
    best = j.groupby(level=0)["conf"].max()
    g["conf"] = best.reindex(g.index).fillna(0).astype(int)
    g["hit"] = g["conf"] > 0
    g["confidence"] = g["conf"].map(CONF_NAME)
    return g


# ---------- 3. buildings, roads, bridges ----------
b = tag_hits(osm["buildings"], BUILDING_BUF_M)
br = tag_hits(osm["bridges"], BRIDGE_BUF_M)
rd = tag_hits(osm["roads"], ROAD_BUF_M)
fp = tag_hits(osm["footpaths"], ROAD_BUF_M)

zone_road = union(imp.buffer(ROAD_BUF_M))
rd_hit = rd[rd["hit"]].copy()
rd_hit["geometry"] = rd_hit.intersection(zone_road)
rd_hit["km_affected"] = rd_hit.length / 1000
km_by_class = rd_hit.groupby("highway")["km_affected"].sum().sort_values(ascending=False).round(2)

stats = {
    "osm_snapshot": SNAP,
    "buildings_total": int(len(b)), "buildings_likely_hit": int(b["hit"].sum()),
    "buildings_hit_high_conf": int((b["conf"] == 3).sum()),
    "road_km_total": round(float(rd.length.sum() / 1000), 1),
    "road_km_likely_affected": round(float(rd_hit["km_affected"].sum()), 2),
    "road_segments_likely_affected": int(rd["hit"].sum()),
    "footpath_segments_likely_affected": int(fp["hit"].sum()),
    "bridges_total": int(len(br)), "bridges_likely_damaged": int(br["hit"].sum()),
    "bridges_damaged_high_conf": int((br["conf"] == 3).sum()),
    "road_km_affected_by_class": km_by_class.to_dict(),
}

# ---------- 4. save for dashboard + cut-off analysis (EPSG:4326 GeoJSON) ----------
def save(gdf, name, cols):
    keep = [c for c in cols if c in gdf.columns] + ["geometry"]
    gdf[keep].to_crs(4326).to_file(f"{RUN}/{name}.geojson", driver="GeoJSON")

save(imp, "impact_polygons", ["impact", "confidence", "area_m2"])
save(b[b["hit"]], "buildings_hit", ["@osmId", "building", "name", "confidence"])
save(rd_hit, "roads_hit", ["@osmId", "highway", "name", "ref", "confidence", "km_affected"])
save(br[br["hit"]], "bridges_hit", ["@osmId", "highway", "bridge", "name", "confidence"])
json.dump(stats, open(f"{RUN}/damage_stats.json", "w"), indent=2)
log["damage"] = stats; json.dump(log, open(f"{RUN}/run_log.json", "w"), indent=2)

print("\n================ DAMAGE SUMMARY (likely hit, pre-event OSM", SNAP, ") ================")
for k, v in stats.items():
    if k != "road_km_affected_by_class":
        print(f"{k:<34} {v}")
print("road km affected by class:", stats["road_km_affected_by_class"])
named = br[br["hit"]].copy()
if len(named):
    named["label"] = named.get("name", pd.Series(index=named.index, dtype=object)).fillna("(unnamed)")
    print("\nBridges likely damaged:")
    for _, r in named.sort_values("conf", ascending=False).iterrows():
        print(f"  - {r['label']:<35} highway={r.get('highway')} confidence={r['confidence']}")

# ---------- 5. map ----------
fig, ax = plt.subplots(1, 2, figsize=(22, 11))
zoom = (314000, 3087000, 324500, 3099000)       # lower Trishuli corridor (UTM 45N, metres)
for axis, title, lim in [(ax[0], "Full area", None), (ax[1], "Zoom: lower Trishuli corridor", zoom)]:
    rd.plot(ax=axis, color="#bbbbbb", linewidth=0.6)
    imp.plot(ax=axis, column="impact", categorical=True, cmap="copper", alpha=0.8, legend=(lim is None))
    if b["hit"].any():
        b[b["hit"]].centroid.plot(ax=axis, color="red", markersize=4, label="building likely hit")
    if len(rd_hit):
        rd_hit.plot(ax=axis, color="black", linewidth=2, label="road affected")
    if br["hit"].any():
        br[br["hit"]].centroid.plot(ax=axis, color="magenta", marker="*", markersize=180, label="bridge likely damaged")
    if lim:
        axis.set_xlim(lim[0], lim[2]); axis.set_ylim(lim[1], lim[3])
    axis.set_title(title); axis.legend(loc="lower right")
plt.tight_layout(); plt.savefig(f"{RUN}/damage_map.png", dpi=110); plt.show()
print("© OpenStreetMap contributors | Contains modified Copernicus Sentinel data 2026.")


In [ ]:
# 7) ACCESS / CUT-OFF
# Track B - who is cut off? Road-network reachability before vs after the flood + repair priority.
# Needs in the run folder (from osm_damage.py): osm_roads/footpaths/bridges/settlements/health/buildings.geojson,
# impact_polygons.geojson, bridges_hit.geojson.   Data: pre-event OSM only.
!pip install -q geopandas networkx shapely

import os, glob, json, datetime as dt
import numpy as np, pandas as pd, geopandas as gpd, networkx as nx, matplotlib.pyplot as plt
from shapely.geometry import LineString, box

RUN = OUT if "OUT" in globals() else sorted(glob.glob("run_*"))[-1]
log = json.load(open(f"{RUN}/run_log.json"))
BBOX = log["bbox"]
SPEED_KMH = {"motorway": 50, "trunk": 40, "primary": 35, "secondary": 30, "tertiary": 25, "unclassified": 20,
             "residential": 20, "service": 15, "track": 12, "path": 4, "footway": 4, "steps": 2, "bridleway": 4}
SNAP_M = 500            # settlement / facility must be within 500 m of a road to be connected
EXIT_M = 300            # road nodes this close to the area edge = "road out of the area"
ROAD_BUF_M, BRIDGE_BUF_M = 10, 30
DETOUR_RATIO = 1.5
SCENARIOS = {"all_detections": {"high", "medium", "low"}, "confident_only": {"high", "medium"}}


def say(msg):
    print(f"[{dt.datetime.now():%H:%M:%S}] {msg}", flush=True)


def load(name):
    p = f"{RUN}/{name}.geojson"
    return gpd.read_file(p) if os.path.exists(p) else gpd.GeoDataFrame(geometry=[], crs=4326)


roads, foot = load("osm_roads"), load("osm_footpaths")
settle, health, bld = load("osm_settlements"), load("osm_health"), load("osm_buildings")
impact, bridges_hit = load("impact_polygons"), load("bridges_hit")
CRS = roads.estimate_utm_crs()
roads, foot, settle, health, bld, impact, bridges_hit = (g.to_crs(CRS) for g in
                                                         (roads, foot, settle, health, bld, impact, bridges_hit))
area = gpd.GeoSeries([box(*BBOX)], crs=4326).to_crs(CRS).iloc[0]


# ---------- 1. graph edges: consecutive vertices of each OSM way (shared OSM nodes = junctions) ----------
def way_edges(gdf, kind):
    rows = []
    g = gdf.explode(index_parts=False)
    for _, r in g.iterrows():
        geom = r.geometry
        if geom is None or geom.geom_type != "LineString":
            continue
        hw = str(r.get("highway") or "unclassified")
        v = SPEED_KMH.get(hw.replace("_link", ""), 15) / 3.6               # m/s
        is_bridge = str(r.get("bridge") or "no") not in ("no", "None", "nan", "")
        cs = [(round(x, 2), round(y, 2)) for x, y in geom.coords]
        for a, b in zip(cs[:-1], cs[1:]):
            if a == b:
                continue
            seg = LineString([a, b])
            rows.append({"u": a, "v": b, "way": str(r.get("@osmId")), "highway": hw, "kind": kind,
                         "bridge": is_bridge, "name": r.get("name"), "len_m": seg.length,
                         "t_s": seg.length / v, "geometry": seg})
    return gpd.GeoDataFrame(rows, crs=CRS)


edges = pd.concat([way_edges(roads, "road"), way_edges(foot, "foot")], ignore_index=True)
edges = gpd.GeoDataFrame(edges, crs=CRS)
say(f"network: {len(edges)} edge segments ({(edges.kind == 'road').sum()} road, {(edges.kind == 'foot').sum()} foot)")

# ---------- 2. which segments are damaged (with the confidence of the impact they touch) ----------
CONF_RANK = {"low": 1, "medium": 2, "high": 3}
zone = impact[["confidence", "geometry"]].copy(); zone["geometry"] = zone.buffer(ROAD_BUF_M)
zone["c"] = zone["confidence"].map(CONF_RANK).fillna(1)
j = gpd.sjoin(edges[["geometry"]], zone[["c", "geometry"]], how="inner", predicate="intersects")
edges["cut_conf"] = j.groupby(level=0)["c"].max().reindex(edges.index).fillna(0).astype(int)
# whole bridge ways flagged by osm_damage.py are cut too
if len(bridges_hit):
    bconf = dict(zip(bridges_hit["@osmId"].astype(str), bridges_hit["confidence"].map(CONF_RANK).fillna(1)))
    bmask = edges["way"].isin(list(bconf))
    edges.loc[bmask, "cut_conf"] = np.maximum(edges.loc[bmask, "cut_conf"], edges.loc[bmask, "way"].map(bconf)).astype(int)
say(f"damaged segments: {(edges.cut_conf > 0).sum()} (any confidence)")


# ---------- 3. graphs + targets ----------
def build_graph(e):
    G = nx.Graph()
    for u, v, t, i in zip(e["u"], e["v"], e["t_s"], e.index):
        if not G.has_edge(u, v) or G[u][v]["t"] > t:
            G.add_edge(u, v, t=t, idx=i)
    return G


nodes_xy = {}
for u, v in zip(edges["u"], edges["v"]):
    nodes_xy[u] = u; nodes_xy[v] = v
node_list = list(nodes_xy)
node_arr = np.array(node_list)
from scipy.spatial import cKDTree
tree = cKDTree(node_arr)


ROAD_EDGES = edges[edges.kind == "road"][["u", "v", "geometry"]]


def snap(points):
    """Nearest ROAD segment within SNAP_M -> its closer end node (works even where road vertices are sparse)."""
    if len(points) == 0:
        return []
    pts = gpd.GeoDataFrame(geometry=list(points), crs=CRS)
    j = gpd.sjoin_nearest(pts, ROAD_EDGES, how="left", max_distance=SNAP_M)
    j = j[~j.index.duplicated()]
    out = []
    for i, p in enumerate(pts.geometry):
        r = j.loc[i]
        if pd.isna(r.get("index_right")):
            out.append(None); continue
        u, v = r["u"], r["v"]
        out.append(u if p.distance(gpd.points_from_xy([u[0]], [u[1]])[0]) <= p.distance(gpd.points_from_xy([v[0]], [v[1]])[0]) else v)
    return out


settle_pts = settle.geometry.representative_point()
settle["node"] = snap(settle_pts)
_blank = pd.Series(index=settle.index, dtype=object)
settle["name_"] = settle.get("name:en", _blank).fillna(settle.get("name", _blank)).fillna("(unnamed)")   # English name if mapped
settle["name_local"] = settle.get("name", _blank)
settle["place"] = settle.get("place", pd.Series(index=settle.index, dtype=object))
health_nodes = [n for n in snap(health.geometry.representative_point()) if n]
town_nodes = [n for n, pl in zip(settle["node"], settle["place"]) if n and pl in ("town", "city")]
node_pts = gpd.GeoSeries(gpd.points_from_xy(node_arr[:, 0], node_arr[:, 1]), crs=CRS)
edge_nodes = [node_list[i] for i in np.where(node_pts.distance(area.exterior).values <= EXIT_M)[0]]
TARGETS = set(health_nodes) | set(town_nodes) | set(edge_nodes)
say(f"targets: {len(set(health_nodes))} health, {len(set(town_nodes))} town nodes, {len(edge_nodes)} roads leaving the area")

# population proxy: buildings within 500 m of each settlement
bc = bld.geometry.representative_point()
btree = cKDTree(np.c_[bc.x, bc.y]) if len(bc) else None
settle["buildings_500m"] = [len(btree.query_ball_point((p.x, p.y), 500)) if btree else 0 for p in settle_pts]


def times(G):
    src = [n for n in TARGETS if n in G]
    return nx.multi_source_dijkstra_path_length(G, src, weight="t") if src else {}


def status_table(e_pre, cut_mask, label):
    G_pre = build_graph(e_pre)
    G_post = build_graph(e_pre[~cut_mask.loc[e_pre.index]])
    t_pre, t_post = times(G_pre), times(G_post)
    st, tp, tq = [], [], []
    for n in settle["node"]:
        a = t_pre.get(n) if n else None
        b = t_post.get(n) if n else None
        tp.append(round(a / 60, 1) if a is not None else None); tq.append(round(b / 60, 1) if b is not None else None)
        if n is None: st.append("no road within 500 m")
        elif a is None: st.append("road not linked pre-event")
        elif b is None: st.append("cut off")
        elif a > 0 and b / a > DETOUR_RATIO and b - a > 300: st.append("long detour")
        else: st.append("unaffected")
    return st, tp, tq, G_pre, G_post, t_post


road_e = edges[edges.kind == "road"]
results = {}
for scen, confs in SCENARIOS.items():
    ranks = {CONF_RANK[c] for c in confs}
    cut = edges["cut_conf"].isin(ranks)
    st, tp, tq, G_pre, G_post, t_post = status_table(road_e, cut, scen)
    settle[f"status_{scen}"] = st
    if scen == "all_detections":
        settle["minutes_pre"], settle["minutes_post"] = tp, tq
        G_post_main, t_post_main, cut_main = G_post, t_post, cut
    results[scen] = pd.Series(st).value_counts().to_dict()

# ---------- 4. footpath check for road-cut-off settlements ----------
all_e = edges
G_foot_post = build_graph(all_e[~cut_main])
t_foot = times(G_foot_post)
settle["reachable_on_foot"] = [bool(n and n in t_foot) for n in settle["node"]]

# ---------- 5. repair priority: reopen one damaged way at a time ----------
cut_off_idx = settle.index[settle["status_all_detections"] == "cut off"]
reached_now = set(t_post_main)
repairs = []
cut_ways = road_e[cut_main.loc[road_e.index]].groupby("way")
say(f"testing {len(cut_ways)} damaged road/bridge ways for repair priority")
for way, grp in cut_ways:
    G_try = G_post_main.copy()
    for _, r in road_e[road_e["way"] == way].iterrows():
        if not G_try.has_edge(r.u, r.v) or G_try[r.u][r.v]["t"] > r.t_s:
            G_try.add_edge(r.u, r.v, t=r.t_s)
    reach = times(G_try)
    fixed = [i for i in cut_off_idx if settle.at[i, "node"] in reach]
    if fixed:
        mid = grp.geometry.union_all().centroid if hasattr(grp.geometry, "union_all") else grp.geometry.unary_union.centroid
        ll = gpd.GeoSeries([mid], crs=CRS).to_crs(4326).iloc[0]
        repairs.append({"way_osm_id": way, "name": grp["name"].dropna().iloc[0] if grp["name"].notna().any() else "(unnamed)",
                        "highway": grp["highway"].iloc[0], "is_bridge": bool(grp["bridge"].any()),
                        "damaged_m": round(grp["len_m"].sum()), "settlements_reconnected": len(fixed),
                        "buildings_reconnected": int(settle.loc[fixed, "buildings_500m"].sum()),
                        "names": ", ".join(settle.loc[fixed, "name_"].head(6)), "lat": round(ll.y, 5), "lon": round(ll.x, 5)})
rep = pd.DataFrame(repairs)
if len(rep):
    rep = rep.sort_values(["settlements_reconnected", "buildings_reconnected"], ascending=False).reset_index(drop=True)
    rep.index += 1
rep.to_csv(f"{RUN}/repairs.csv", index_label="rank")

# ---------- 6. save + summary ----------
out = settle.copy(); out["geometry"] = settle_pts
keep = ["name_", "name_local", "place", "status_all_detections", "status_confident_only", "minutes_pre", "minutes_post",
        "reachable_on_foot", "buildings_500m", "geometry"]
out[keep].rename(columns={"name_": "name"}).to_crs(4326).to_file(f"{RUN}/settlements_status.geojson", driver="GeoJSON")
stats = {"settlements": int(len(settle)), "scenarios": results,
         "cut_off_but_reachable_on_foot": int(((settle.status_all_detections == "cut off") & settle.reachable_on_foot).sum()),
         "buildings_near_cut_off_settlements": int(settle.loc[settle.status_all_detections == "cut off", "buildings_500m"].sum()),
         "repair_items_that_reconnect_someone": int(len(rep))}
json.dump(stats, open(f"{RUN}/access_stats.json", "w"), indent=2)
log["access"] = stats; json.dump(log, open(f"{RUN}/run_log.json", "w"), indent=2)

print("\n================ ACCESS SUMMARY ================")
for scen, counts in results.items():
    print(f"{scen:<16} {counts}")
print(f"cut off by road but reachable on foot: {stats['cut_off_but_reachable_on_foot']}")
print(f"buildings within 500 m of cut-off settlements (population proxy): {stats['buildings_near_cut_off_settlements']}")
co = settle[settle.status_all_detections.isin(["cut off", "long detour"])].sort_values("buildings_500m", ascending=False)
print("\nMost affected settlements (all detections):")
for _, r in co.head(15).iterrows():
    print(f"  {r['name_']:<28} {r['place'] or '':<9} {r['status_all_detections']:<12} "
          f"before {r['minutes_pre']} min -> after {r['minutes_post']} min | foot: {'yes' if r['reachable_on_foot'] else 'no'} "
          f"| ~{r['buildings_500m']} buildings")
print("\nTOP REPAIRS (reopen this -> reconnects):")
print(rep.head(10)[["name", "highway", "is_bridge", "settlements_reconnected", "buildings_reconnected", "names"]].to_string()
      if len(rep) else "  none - no settlement is fully cut off in this scenario")

# ---------- 7. map ----------
colors = {"cut off": "#d00000", "long detour": "#f08000", "unaffected": "#2a9d3a",
          "road not linked pre-event": "#777777", "no road within 500 m": "#bbbbbb"}
import warnings; warnings.filterwarnings("ignore", message=".*Glyph.*"); warnings.filterwarnings("ignore", message=".*Devanagari.*")
fig, ax = plt.subplots(figsize=(13, 14))
road_e.plot(ax=ax, color="#c8c8c8", linewidth=0.6)
road_e[cut_main.loc[road_e.index]].plot(ax=ax, color="black", linewidth=2.5)
impact.plot(ax=ax, color="#8b4513", alpha=0.5)
for s, c in colors.items():
    sub = settle[settle.status_all_detections == s]
    if len(sub):
        gpd.GeoSeries(sub.geometry.representative_point(), crs=CRS).plot(ax=ax, color=c, markersize=60, edgecolor="white",
                                                                          label=f"{s} ({len(sub)})")
for _, r in co.head(12).iterrows():
    p = r.geometry.representative_point(); ax.annotate(r["name_"], (p.x, p.y), fontsize=8, xytext=(4, 4), textcoords="offset points")
if len(health):
    health.geometry.representative_point().plot(ax=ax, color="blue", marker="P", markersize=90, label="health facility")
ax.legend(loc="lower right"); ax.set_title("Settlement access after the flood (all detections)")
plt.tight_layout(); plt.savefig(f"{RUN}/access_map.png", dpi=110); plt.show()
print("© OpenStreetMap contributors | Contains modified Copernicus Sentinel data 2026.")


In [ ]:
# 8) Save everything (Colab disk is temporary)
import shutil
from google.colab import files
shutil.make_archive("trackb_run", "zip", OUT)
files.download("trackb_run.zip")
print("Downloading trackb_run.zip - contains impact map, damage + access GeoJSON, stats, run_log")